# MSE 232 Project - Group 13
## Optimizing the 2026 FIFA World Cup Group-Stage Schedule

# 1. Data Import and Preparation

## This section imports the datasets and creates the sets and parameters used in the optimization models. The project data is stored in separate CSV files so that the optimization model can be updated without changing the model formulation.

In [ ]:
# libraries
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB
import matplotlib.pyplot as plt

# csv files
teams = pd.read_csv("teams.csv")
venues = pd.read_csv("venues.csv")
matches = pd.read_csv("matches.csv")
travel = pd.read_csv("travel.csv")
weather = pd.read_csv("weather.csv")
audience_regions = pd.read_csv("audience_regions.csv")

In [ ]:
# data inspection

print("Teams:", teams.shape)
print("Venues:", venues.shape)
print("Matches:", matches.shape)
print("Travel:", travel.shape)
print("Weather:", weather.shape)
print("Audience Regions:", audience_regions.shape)

display(teams.head())
display(venues.head())
display(matches.head())
display(travel.head())

print("Number of teams:", teams["team_id"].nunique())
print("Number of venues:", venues["venue_id"].nunique())
print("Number of matches:", matches["match_id"].nunique())
print("Number of travel combinations:", len(travel))

In [ ]:
# data formatting

matches["match_date"] = pd.to_datetime(matches["match_date"])
weather["date"] = pd.to_datetime(weather["date"])

teams["team_id"] = teams["team_id"].astype(int)
venues["venue_id"] = venues["venue_id"].astype(int)

matches["match_id"] = matches["match_id"].astype(int)
matches["team_1_id"] = matches["team_1_id"].astype(int)
matches["team_2_id"] = matches["team_2_id"].astype(int)
matches["actual_venue_id"] = matches["actual_venue_id"].astype(int)

travel["team_id"] = travel["team_id"].astype(int)
travel["venue_id"] = travel["venue_id"].astype(int)

In [ ]:
# defining sets

Teams = teams["team_id"].tolist()
Venues = venues["venue_id"].tolist()
Matches = matches["match_id"].tolist()

Groups = sorted(matches["group"].unique().tolist())
Dates = sorted(matches["match_date"].unique().tolist())

Times = list(range(10, 23))

In [ ]:
# match parameters

match_group = dict(zip(
    matches["match_id"],
    matches["group"]
))

group_round = dict(zip(
    matches["match_id"],
    matches["group_round"]
))

team_1 = dict(zip(
    matches["match_id"],
    matches["team_1_id"]
))

team_2 = dict(zip(
    matches["match_id"],
    matches["team_2_id"]
))

match_date = dict(zip(
    matches["match_id"],
    matches["match_date"]
))

actual_venue = dict(zip(
    matches["match_id"],
    matches["actual_venue_id"]
))

actual_kickoff_utc = dict(zip(
    matches["match_id"],
    matches["actual_kickoff_utc"]
))

In [ ]:
# team parameters

team_name = dict(zip(
    teams["team_id"],
    teams["team_name"]
))

base_camp_utc_offset = dict(zip(
    teams["team_id"],
    teams["base_camp_utc_offset"]
))

In [ ]:
# venue parameters

venue_name = dict(zip(
    venues["venue_id"],
    venues["venue_name"]
))

venue_city = dict(zip(
    venues["venue_id"],
    venues["city"]
))

venue_country = dict(zip(
    venues["venue_id"],
    venues["country"]
))

venue_utc_offset = dict(zip(
    venues["venue_id"],
    venues["utc_offset"]
))

venue_capacity = dict(zip(
    venues["venue_id"],
    venues["capacity"]
))

In [ ]:
# travel parameters

travel_time = {}

for _, row in travel.iterrows():
    i = row["team_id"]
    v = row["venue_id"]

    travel_time[(i, v)] = row["total_time_travelled_one_way_hours"]

travel_distance = {}

for _, row in travel.iterrows():
    i = row["team_id"]
    v = row["venue_id"]

    travel_distance[(i, v)] = row["total_distance_travelled_one_way_km"]

# missing travel check

missing_travel = []

for i in Teams:
    for v in Venues:
        if (i, v) not in travel_time:
            missing_travel.append((i, v))

print("Missing travel combinations:", len(missing_travel))

In [ ]:
# helper sets

# matches by date

matches_by_date = {}

for d in Dates:
    matches_by_date[d] = []

    for m in Matches:
        if match_date[m] == d:
            matches_by_date[d].append(m)

# matches for each team
team_matches = {}

for i in Teams:
    team_matches[i] = []

    for m in Matches:
        if team_1[m] == i or team_2[m] == i:
            team_matches[i].append(m)

for i in Teams:
    team_matches[i] = sorted(
        team_matches[i],
        key=lambda m: match_date[m]
    )

# matches by round
Round1Matches = []
Round2Matches = []
Round3Matches = []

for m in Matches:

    if group_round[m] == 1:
        Round1Matches.append(m)

    elif group_round[m] == 2:
        Round2Matches.append(m)

    elif group_round[m] == 3:
        Round3Matches.append(m)

# matches by group and round
matches_by_group_round = {}

for g in Groups:
    for r in [1, 2, 3]:

        matches_by_group_round[(g, r)] = []

        for m in Matches:
            if match_group[m] == g and group_round[m] == r:
                matches_by_group_round[(g, r)].append(m)

# 2. Metric Preparation and Evaluation Parameters

## This section defines the additional parameters, functions, and lookup tables used to evaluate both the FIFA schedule and the optimized schedules. Defining these measures before evaluating any schedule ensures that FIFA and all optimization models are assessed using the same methodology.

In [ ]:
# additional venue parameters

weather_protected = dict(zip(
    venues["venue_id"],
    venues["weather_protected"]
))

roof_type = dict(zip(
    venues["venue_id"],
    venues["roof_type"]
))

In [ ]:
# heat index function

def calculate_heat_index(temp_f, humidity):

    simple_hi = 0.5 * (
        temp_f
        + 61
        + 1.2 * (temp_f - 68)
        + 0.094 * humidity
    )

    simple_hi = (simple_hi + temp_f) / 2

    if simple_hi < 80:
        return simple_hi

    hi = (
        -42.379
        + 2.04901523 * temp_f
        + 10.14333127 * humidity
        - 0.22475541 * temp_f * humidity
        - 0.00683783 * temp_f**2
        - 0.05481717 * humidity**2
        + 0.00122874 * temp_f**2 * humidity
        + 0.00085282 * temp_f * humidity**2
        - 0.00000199 * temp_f**2 * humidity**2
    )

    return hi

In [ ]:
# heat index for weather data

weather["heat_index_f"] = weather.apply(
    lambda row: calculate_heat_index(
        row["temperature_f"],
        row["humidity_pct"]
    ),
    axis=1
)

In [ ]:
# weather lookup

weather_lookup = {}

for _, row in weather.iterrows():

    key = (
        row["venue_id"],
        row["date"],
        row["hour_local"]
    )

    weather_lookup[key] = row["heat_index_f"]

In [ ]:
# time zone difference lookup

timezone_difference = {}

for i in Teams:
    for v in Venues:

        timezone_difference[(i, v)] = abs(
            base_camp_utc_offset[i]
            - venue_utc_offset[v]
        )

# 3. FIFA Schedule Baseline Analysis
## This section evaluates FIFA's actual group-stage schedule using the common metrics defined above. Because the FIFA venue and kickoff assignments are fixed, no optimization is performed in this section. The existing FIFA schedule is evaluated directly and used as the benchmark for comparison with the optimized schedules.

In [ ]:
# constructing fifa schedule

fifa_schedule = matches[
    [
        "match_id",
        "group",
        "group_round",
        "team_1_id",
        "team_2_id",
        "match_date",
        "actual_venue_id",
        "actual_kickoff_utc"
    ]
].copy()

fifa_schedule["team_1_name"] = fifa_schedule["team_1_id"].map(team_name)
fifa_schedule["team_2_name"] = fifa_schedule["team_2_id"].map(team_name)

fifa_schedule["venue_name"] = fifa_schedule["actual_venue_id"].map(venue_name)
fifa_schedule["venue_city"] = fifa_schedule["actual_venue_id"].map(venue_city)
fifa_schedule["venue_country"] = fifa_schedule["actual_venue_id"].map(venue_country)

In [ ]:
# adding world cup 2026 actual kickoff times

fifa_schedule["kickoff_datetime_utc"] = (
    fifa_schedule["match_date"]
    + pd.to_timedelta(
        fifa_schedule["actual_kickoff_utc"],
        unit="h"
    )
)

In [ ]:
# fifa schedule travel analysis

fifa_total_travel_time = 0

for m in Matches:

    i = team_1[m]
    j = team_2[m]
    v = actual_venue[m]

    fifa_total_travel_time += 2 * travel_time[(i, v)]
    fifa_total_travel_time += 2 * travel_time[(j, v)]

print(
    "FIFA total team travel time:",
    round(fifa_total_travel_time, 2),
    "hours"
)

In [ ]:
# fifa travel by team
fifa_team_travel = {}

for i in Teams:

    fifa_team_travel[i] = 0

    for m in team_matches[i]:

        v = actual_venue[m]

        fifa_team_travel[i] += (
            2 * travel_time[(i, v)]
        )

fifa_team_travel_df = pd.DataFrame({
    "team_id": Teams,
    "team_name": [team_name[i] for i in Teams],
    "total_travel_time_hours": [
        fifa_team_travel[i]
        for i in Teams
    ]
})

In [ ]:
# summary metrics for fifa schedule

fifa_avg_team_travel = (
    fifa_team_travel_df["total_travel_time_hours"].mean()
)

fifa_max_team_travel = (
    fifa_team_travel_df["total_travel_time_hours"].max()
)

fifa_min_team_travel = (
    fifa_team_travel_df["total_travel_time_hours"].min()
)

fifa_std_team_travel = (
    fifa_team_travel_df["total_travel_time_hours"].std()
)

In [ ]:
# fifa rest and effective recovery

fifa_recovery_rows = []

for i in Teams:

    matches_i = team_matches[i]

    for k in range(len(matches_i) - 1):

        previous_match = matches_i[k]
        next_match = matches_i[k + 1]

        previous_row = fifa_schedule[
            fifa_schedule["match_id"] == previous_match
        ].iloc[0]

        next_row = fifa_schedule[
            fifa_schedule["match_id"] == next_match
        ].iloc[0]

        previous_kickoff = previous_row["kickoff_datetime_utc"]
        next_kickoff = next_row["kickoff_datetime_utc"]

        raw_rest_hours = (
            next_kickoff - previous_kickoff
        ).total_seconds() / 3600

        previous_venue = previous_row["actual_venue_id"]
        next_venue = next_row["actual_venue_id"]

        recovery_travel_time = (
            travel_time[(i, previous_venue)]
            + travel_time[(i, next_venue)]
        )

        effective_recovery_hours = (
            raw_rest_hours
            - recovery_travel_time
        )

        fifa_recovery_rows.append({
            "team_id": i,
            "team_name": team_name[i],
            "previous_match": previous_match,
            "next_match": next_match,
            "raw_rest_hours": raw_rest_hours,
            "recovery_travel_time_hours": recovery_travel_time,
            "effective_recovery_hours": effective_recovery_hours
        })

fifa_recovery = pd.DataFrame(fifa_recovery_rows)

In [ ]:
# summary metrics for fifa recovery

fifa_min_raw_rest = fifa_recovery["raw_rest_hours"].min()
fifa_avg_raw_rest = fifa_recovery["raw_rest_hours"].mean()

fifa_min_effective_recovery = (
    fifa_recovery["effective_recovery_hours"].min()
)

fifa_avg_effective_recovery = (
    fifa_recovery["effective_recovery_hours"].mean()
)

fifa_recovery["meets_72_hour_raw_rest"] = (
    fifa_recovery["raw_rest_hours"] >= 72
)

print("Minimum FIFA raw rest:", round(fifa_min_raw_rest, 2))
print("Average FIFA raw rest:", round(fifa_avg_raw_rest, 2))

print(
    "Minimum FIFA effective recovery:",
    round(fifa_min_effective_recovery, 2)
)

print(
    "Average FIFA effective recovery:",
    round(fifa_avg_effective_recovery, 2)
)

print(
    "Intervals below 72 hours raw rest:",
    (~fifa_recovery["meets_72_hour_raw_rest"]).sum()
)

In [ ]:
# fifa schedule time zone analysis

fifa_timezone_rows = []

for m in Matches:

    v = actual_venue[m]

    for i in [team_1[m], team_2[m]]:

        fifa_timezone_rows.append({
            "match_id": m,
            "team_id": i,
            "team_name": team_name[i],
            "venue_id": v,
            "timezone_difference_hours":
                timezone_difference[(i, v)]
        })

fifa_timezone = pd.DataFrame(fifa_timezone_rows)

fifa_avg_timezone_difference = (
    fifa_timezone["timezone_difference_hours"].mean()
)

fifa_max_timezone_difference = (
    fifa_timezone["timezone_difference_hours"].max()
)

In [ ]:
# fifa schedule weather analysis

fifa_schedule["kickoff_local_hour"] = (
    fifa_schedule["actual_kickoff_utc"]
    + fifa_schedule["actual_venue_id"].map(
        venue_utc_offset
    )
) % 24

fifa_weather_rows = []

for _, row in fifa_schedule.iterrows():

    m = row["match_id"]
    v = row["actual_venue_id"]
    d = row["match_date"]
    h = row["kickoff_local_hour"]

    heat_index = weather_lookup[(v, d, h)]

    if weather_protected[v] == 1:
        weather_exposure = 0
    else:
        weather_exposure = heat_index

    fifa_weather_rows.append({
        "match_id": m,
        "venue_id": v,
        "date": d,
        "local_hour": h,
        "heat_index_f": heat_index,
        "weather_protected": weather_protected[v],
        "weather_exposure": weather_exposure
    })
    
fifa_weather = pd.DataFrame(fifa_weather_rows)

fifa_total_weather_exposure = (
    fifa_weather["weather_exposure"].sum()
)

fifa_avg_weather_exposure = (
    fifa_weather["weather_exposure"].mean()
)

fifa_max_weather_exposure = (
    fifa_weather["weather_exposure"].max()
)

In [ ]:
# fifa schedule seating opportunity

fifa_seating_opportunity = 0

for m in Matches:
    v = actual_venue[m]
    fifa_seating_opportunity += venue_capacity[v]

In [ ]:
# fifa schedule venue and country usage

fifa_venue_usage = {}

for v in Venues:
    fifa_venue_usage[v] = 0

for m in Matches:
    v = actual_venue[m]
    fifa_venue_usage[v] += 1

fifa_country_usage = {}

for m in Matches:

    v = actual_venue[m]
    country = venue_country[v]

    if country not in fifa_country_usage:
        fifa_country_usage[country] = 0

    fifa_country_usage[country] += 1

In [ ]:
# fifa schedule baseline summary

fifa_baseline = {
    "total_travel_time_hours": fifa_total_travel_time,
    "average_team_travel_time_hours": fifa_avg_team_travel,
    "maximum_team_travel_time_hours": fifa_max_team_travel,
    "minimum_team_travel_time_hours": fifa_min_team_travel,
    "travel_time_std_hours": fifa_std_team_travel,

    "minimum_raw_rest_hours": fifa_min_raw_rest,
    "average_raw_rest_hours": fifa_avg_raw_rest,

    "minimum_effective_recovery_hours":
        fifa_min_effective_recovery,

    "average_effective_recovery_hours":
        fifa_avg_effective_recovery,

    "average_timezone_difference_hours":
        fifa_avg_timezone_difference,

    "maximum_timezone_difference_hours":
        fifa_max_timezone_difference,

    "total_weather_exposure":
        fifa_total_weather_exposure,

    "average_weather_exposure":
        fifa_avg_weather_exposure,

    "maximum_weather_exposure":
        fifa_max_weather_exposure,

    "seating_opportunity":
        fifa_seating_opportunity
}

fifa_baseline_df = pd.DataFrame(
    fifa_baseline.items(),
    columns=["metric", "value"]
)

display(fifa_baseline_df)

## 4. Model 1 - Travel Optimization

## 5. Model 2 - Player Welfare Optimization

## 6. Model 3 - Operational and Commercial Optimization

## 7. Model 4 - Goal Optimization

## 8. Model Comparison and Sensitivity Analysis

## 9. Final Recommended Schedule